# FINM 32000: Homework 2

By Andrew McLaughlin

## Problem 1

Consider a particular stock index that is defined to have value equal to the price of a fixed basket of non-dividend-paying stocks. Suppose that it follows the Black-Scholes dynamics with $\sigma = 0.4$, and that the time-0 index level is $S_0 = 100$. Consider a three-month ($=0.25$ year) up-and-out European put, struck at 95, with a discretely monitored knock-out barrier at 114, observed at times $0.02, 0.04, \ldots, 0.24$. That is, our option knocks out if and only if the index is at or above 114 at an observation time (where the unit of time in this course will always be years, unless otherwise indicated). Let the constant risk-free interest rate be $r = 0$.

In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import bisect, brentq
from copy import copy

In [2]:
class UpAndOutPut:

    def __init__(self, K, T, barrier, observationinterval):
        self.K = K
        self.T = T
        self.barrier = barrier
        self.observationinterval = observationinterval

In [3]:
#note the template notebook had the barrier at 107 but the homework pdf requests 114
hw2contract = UpAndOutPut(K=95, T=0.25, barrier=114, observationinterval=0.02)

In [4]:
#hwk 1 GBMdyanmics, changed X to S to reflect hw2dynamics
class GBMdynamics:

    def __init__(self, S, r, rGrow, sigma=None):
        self.S = S
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

In [5]:
# Use the same GBMdynamics class from HW 1

hw2dynamics = GBMdynamics(S=100, sigma=0.4, rGrow=0, r=0)
hw2dynamics

### (a)

Write a Python function to price our option at time 0 using a trinomial tree with probabilities specified in L2.21. Some of the code is already provided for you.

The provided code uses the time step $\Delta t = T/N$ as suggested in class. We will want the barrier-monitoring times to be represented in the tree, preferably without introducing unequal time intervals anywhere, so we will want to choose $N$ a multiple of 25. Your code should be able to accept any such $N$; a user who desires high accuracy can choose $N$ large; a user who desires high speed can choose $N$ small. For FINM 32000, please report a price using $N$ chosen large enough that your output has converged, in your judgment (no proof needed), to within $0.01 of the true price. In this example, $N = 100$ will not be sufficient.

The provided code chooses the space step $\Delta x$ such that the log of the barrier level $H = 114$ is exactly halfway between consecutive log price levels of the tree. Subject to this constraint, it chooses $\Delta x$ close to the recommended $\sigma \sqrt{3\Delta t}$ value. In other words, the constraint is that there exists an integer $j$ such that $\log(114)$ is halfway between the $j$th and $(j+1)$th log-price levels:

$$
\log S_0 + (j + 0.5)\Delta x = \log H.
$$

And the integer $j$ is chosen such that the $\Delta x$ which satisfies the constraint is approximately $\sigma \sqrt{3\Delta t}$, so we take $j$ to be the nearest integer to

$$
\frac{\log(H/S_0)}{\sigma \sqrt{3\Delta t}} - 0.5.
$$

Why do we have this "halfway between" requirement? If you try instead putting $\log H$ at a log-price level, you will find the accuracy to be worse than the "halfway between" procedure, for this discretely monitored barrier option.

In [6]:
class TreeEngine:

    def __init__(self, N):
        self.N = N

    def price_upandout(self, dynamics, contract):

        deltat = contract.T / self.N
        J = np.ceil(np.log(contract.barrier/dynamics.S)/(dynamics.sigma*np.sqrt(3*deltat))-0.5)
        deltax = np.log(contract.barrier/dynamics.S)/(J+0.5)

        Sgrid = dynamics.S*np.exp(np.linspace(self.N, -self.N, num=2*self.N+1, endpoint=True)*deltax)
        #Here I decided to make the SMALLER indexes in this array correspond to HIGHER S

        numTimestepsPerObs = contract.observationinterval/deltat
        if abs(numTimestepsPerObs-round(numTimestepsPerObs)) > 1e-8:
            raise ValueError("This value of N fails to place the observation dates in the tree.")

        nu =  dynamics.rGrow - dynamics.sigma**2 * .5      # complete this

        A = (dynamics.sigma**2 * deltat + nu**2 * deltat**2) / deltax**2
        B = nu * deltat / deltax


        Pu = .5 * (A + B)    # complete this
        Pd = .5 * (A - B)    # complete this
        Pm = 1 - A           # complete this

        optionprice = np.maximum(contract.K-Sgrid,0)   #an array of time-T option prices.

        #Next, induct backwards to time 0, updating the optionprice array
        #Hint: if x is an array, then what are x[2:] and x[1:-1] and x[:-2]

        for t in np.linspace(self.N-1, 0, num=self.N, endpoint=True)*deltat:
            # insert lines of code here if needed
            C_u = optionprice[:-2]
            C_m = optionprice[1:-1]
            C_d = optionprice[2:]
            optionprice = np.exp(-1 * dynamics.r * deltat) * (Pu * C_u + Pm * C_m + Pd * C_d) #complete this
            Sgrid = Sgrid[1:-1]
            step = int(round(t / deltat))
            if step % int(round(numTimestepsPerObs)) == 0 and step != 0:
                optionprice[Sgrid >= contract.barrier] = 0

        return optionprice[0]
        #The [0] is assuming that we are shrinking the optionprice array in each iteration of the loop,
        #until finally there is only 1 element in the array.
        #If instead you are keeping unchanged the size of the optionprice array in each iteration,
        #then you need to change the [0] to a different index.

    
    def price_euro(self, dynamics, contract):
        #this function is a copy of above but ignores the barrier so that a vanilla option price can be calculated
        deltat = contract.T / self.N
        J = np.ceil(np.log(contract.barrier/dynamics.S)/(dynamics.sigma*np.sqrt(3*deltat))-0.5)
        deltax = np.log(contract.barrier/dynamics.S)/(J+0.5)

        Sgrid = dynamics.S*np.exp(np.linspace(self.N, -self.N, num=2*self.N+1, endpoint=True)*deltax)
        #Here I decided to make the SMALLER indexes in this array correspond to HIGHER S

        numTimestepsPerObs = contract.observationinterval/deltat
        if abs(numTimestepsPerObs-round(numTimestepsPerObs)) > 1e-8:
            raise ValueError("This value of N fails to place the observation dates in the tree.")

        nu =  dynamics.rGrow - dynamics.sigma**2 * .5      # complete this

        A = (dynamics.sigma**2 * deltat + nu**2 * deltat**2) / deltax**2
        B = nu * deltat / deltax


        Pu = .5 * (A + B)    # complete this
        Pd = .5 * (A - B)    # complete this
        Pm = 1 - A           # complete this

        optionprice = np.maximum(contract.K-Sgrid,0)   #an array of time-T option prices.

        #Next, induct backwards to time 0, updating the optionprice array
        #Hint: if x is an array, then what are x[2:] and x[1:-1] and x[:-2]

        for t in np.linspace(self.N-1, 0, num=self.N, endpoint=True)*deltat:
            # insert lines of code here if needed
            C_u = optionprice[:-2]
            C_m = optionprice[1:-1]
            C_d = optionprice[2:]
            optionprice = np.exp(-1 * dynamics.r * deltat) * (Pu * C_u + Pm * C_m + Pd * C_d) #complete this
        #    Sgrid = Sgrid[1:-1]
        #    step = int(round(t / deltat))
        #    if step % int(round(numTimestepsPerObs)) == 0 and step != 0:
        #        optionprice[Sgrid >= contract.barrier] = 0

        return optionprice[0]  


In [7]:
hw2tree=TreeEngine(N=100)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.311979858729038)

In [8]:
hw2tree=TreeEngine(N=500)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.304324358395272)

In [9]:
hw2tree=TreeEngine(N=1000)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.301546106266766)

In [10]:
hw2tree=TreeEngine(N=10000)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.301072607079835)

**Answer** <br>

I tested several valid values of N and found that the price stabilizes near $5.30. In particular, increasing N from 1000 to 10000 changes the price by less than $0.01, so N = 1000 is sufficient for the required level of accuracy.


Regarding the "halfway between" requirement: 

Because the option value is discontinuous at the barrier, placing the barrier exactly on a tree node forces that node to be treated as fully knocked out, even though the node is only an approximation to a range of nearby outcomes. That creates an asymmetric approximation error near the barrier. By placing the barrier halfway between two adjacent log-price nodes, one node lies clearly below the barrier and the next lies clearly above it, so the discontinuity is represented between nodes rather than on a node. This gives a more balanced and typically more accurate approximation of the barrier option’s value.

### (b)

Consider an up-and-in put with the same terms. Specifically, this option has the same strike, expiry, barrier, and monitoring dates, but it pays at expiry the put payoff only if the index was at or above the 114 knock-in barrier at some monitoring date; otherwise, it pays nothing.

Using your part (a) result, find the time-0 price of the up-and-in put.

In [ ]:
#If we know the price of a vanilla-put and understand an up and in put + an up and out put = vanilla put, then we can solve

# upandout = vanillaput - upandin

tree_upandout=TreeEngine(N=1000)

price_upandout = tree_upandout.price_upandout(hw2dynamics, hw2contract)
price_euro = tree_upandout.price_euro(hw2dynamics, hw2contract) #price_euro() ignores barrier

price_upandin = price_euro - price_upandout

price_upandin


np.float64(0.217341132747511)

**Answer** <br>

The price of the up and in put is $0.22$

### (c)

Consider a continuously monitored barrier option paying at time $T = 0.25$ the amount

$$
(95 - S_{0.25})^+ \mathbf{1}_{\max_{0 \le t \le 0.25} S_t < 114},
$$

where the indicator variable $\mathbf{1}(A) := \mathbf{1}_A := 1$ if event $A$ occurs, 0 otherwise.

#### (c1)

Is the time-0 price of the continuously monitored barrier option greater than or smaller than the time-0 price of the discretely monitored option in (a)? Justify briefly without doing any numerical calculations. One sentence is enough.

**Answer** <br>
The continuously monitored option is worth less as any path that survives under continuous must survive under discrete, but not vice versa.

#### (c2)

The continuously monitored barrier option can be replicated by a portfolio of $T$-expiry options, long 1 plain vanilla put struck at 95, and short $\alpha$ plain vanilla calls struck at 136.8.

The replication strategy is as follows. If $S$ does not hit the barrier before time $T$, then simply collect the time-$T$ payout of the 95 put, as desired. If $S$ does hit the barrier, then at the time when $S$ is at the barrier, the 1 unit of the vanilla put has value that exactly cancels the value of the $-\alpha$ units of the plain vanilla call; so at that time, we close out the portfolio positions, for a net payment of zero, as desired.

Solve analytically for the quantity $\alpha$ that makes this replication strategy valid, and find the time-0 value of the continuously monitored barrier option. Do not use a tree.

**Answer: finding alpha** <br>
The portfolio:

- 1 vanilla put with strike 95
- -$\alpha$ vanila calls withs strike 136.8

We will consider the payoff at different cases to understand the value of $\alpha$:

1. The barrier is never hit.
    -  The call expires worthless and all that remains is the value of the vanilla put
    - Payoff $(95 - S_t)^+$, which is equivalent to the barrier option.

2. Barrier is hit at some time $t < T$
    - There exists some $S_t = 114$ with time remaining to maturity $u = T - t$
    - We can use black scholes:
    - $C(114, 136.8, u) = 114N(d_1) - 136.8N(d_2)$
    - $P(114, 95, u) = 95N(-d_2) - 114N(-d_1)$
    - when the stock is at the barrier $S=114$, the Black-Scholes $d$-terms for the call line up with those of the put so $d_1^{\text{call}} = -d_2^{\text{put}}$, $d_2^{\text{call}} = -d_1^{\text{put}}$
    - $C(114, 136.8, u) = 114N(-d_2) - 136.8N(-d_1)$
    - Now we can set up the final equation:

        - $P(114, 95, u) = {\alpha}C(114, 136.8, u)$  
        - $95N(-d_2) - 114N(-d_1) = \alpha (114N(-d_2) - 136.8N(-d_1))$ 
        - Comparing the coefficient on the first term:  
        - $95 = {\alpha}*114$ 
        - $\alpha = \frac{95}{114}$ 

Hence, the replicate the barrier option $\alpha = \frac{95}{114}$ 




In [ ]:
#pricing using the hwk 1 code, truncated to only use BS pricer
class CallOption:

    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price

class AnalyticEngine:

    def __init__(self):
        pass

    def BSpriceCall(self, dynamics, contract):
        # Ignores contract.price if given, because this function calculates price based on the dynamics.
        # Returns time-0 price.

        F = dynamics.S*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        d2 = d1-std
        return np.exp(-dynamics.r*contract.T)*(F*norm.cdf(d1)-contract.K*norm.cdf(d2))

In [ ]:
hw2analytic = AnalyticEngine()

call_price = hw2analytic.BSpriceCall(GBMdynamics(sigma=.4, rGrow=0, S=100, r=0),
               CallOption(K=136.8, T=0.25))

put_price = hw2analytic.BSpriceCall(GBMdynamics(sigma=.4, rGrow=0, S=100, r=0),
               CallOption(K=95, T=0.25)) - 100 + 95 #put-call parity pricing for the put

alpha = 95/114

barrier_replication_price = put_price - alpha * call_price 

barrier_replication_price

np.float64(5.0315264305740595)

**Answer: Time-0 Price** <br>
The time zero price, using the hwk 1 BS price function, provides us a time 0 orice of approx. $5.03.

## Problem 2

### (a)

Interest rate is 0. A non-dividend-paying stock $S$ has time-0 price $S_0 = 100$. At time 0, you observe the dollar prices of at-the-money ($K = 100$) European calls on $S$ at 0.5-year and 1-year expiries to be 11.25 and 12.00, respectively.

Find the time-0 Black-Scholes implied volatilities of these two options.

In [13]:
class CallOption:

    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price

class AnalyticEngine:

    def __init__(self):
        pass

    def BSpriceCall(self, dynamics, contract):
        # Ignores contract.price if given, because this function calculates price based on the dynamics.
        # Returns time-0 price.

        F = dynamics.S*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        d2 = d1-std
        return np.exp(-dynamics.r*contract.T)*(F*norm.cdf(d1)-contract.K*norm.cdf(d2))

    def BSvega(self, dynamics, contract):
        # Returns time-$0$ vega

        F = dynamics.S*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        return np.exp(-dynamics.r*contract.T)*F*norm.pdf(d1)*np.sqrt(contract.T)

    def IV(self, dynamics, contract):
        # ignores dynamics.sigma, because this function solves for sigma.
        # Returns time-$0$ implied volatility

        if contract.price is None:
            raise ValueError('Contract price must be given')

        df = np.exp(-dynamics.r*contract.T)  #discount factor
        F = dynamics.S*np.exp(dynamics.rGrow*contract.T)
        lowerbound = np.max([0, df*(F - contract.K)])
        C = contract.price
        if C < lowerbound:
            return np.nan
        if C == lowerbound:
            return 0
        if C >= F*df:
            return np.nan

        dynamics_try = copy(dynamics)
        # We "try" values of sigma until we find sigma that generates price C

        # First find lower and upper bounds
        sigma_try = 0.2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) > C:
            sigma_try /= 2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) < C:
            sigma_try *= 2
        hi = sigma_try
        lo = hi/2
        # We have calculated "lo" and "hi" which bound the implied volatility from below and above.
        # In other words, the implied volatility is somewhere in the interval [lo,hi].
        # Then, to calculate the implied volatility within that interval,
        # for purposes of this homework, you may either (A) write your own bisection algorithm,
        # or (B) use scipy.optimize.bisect or (C) use scipy.optimize.brentq
        # You will need to provide lo and hi to those solvers.
        # There are other solvers that do not require you to bound the solution
        # from below and above (for instance, scipy.optimize.fsolve is a useful solver).
        # However, if you are able to bound the solution (of a single-variable problem),
        # then bisection or Brent will be more reliable.

        # Use bisect or brentq imported from scipy.optimize, or write your own bisection algorithm
        impliedVolatility = brentq(lambda sigma_guess: self.BSpriceCall(dynamics_try.update_sigma(sigma_guess), contract)-C,lo, hi)
        return impliedVolatility

In [14]:
hw2analytic = AnalyticEngine()

hw2analytic.IV(GBMdynamics(sigma=None, rGrow=0, S=100, r=0),
               CallOption(K=100, T=0.5, price=11.25))

0.4001327809210663

In [15]:
hw2analytic.IV(GBMdynamics(sigma=None, rGrow=0, S=100, r=0),
               CallOption(K=100, T=1, price=12))

0.3019384309935543

**Answer** <br>

$IV_{t=0, T=0.5} = 0.4$

$IV_{t=0, T=0.75} = 0.3$

### (b)

Consider an at-the-money European call on $S$ with expiry 0.75. Suppose that you try to price it by assuming that its implied volatility is equal to the midpoint, the arithmetic average, of the 0.5-expiry and the 1.0-expiry implied volatilities. Under that assumption, what would be the time-0 price of the 0.75-expiry call?

In [16]:
sixmonth_IV = hw2analytic.IV(GBMdynamics(sigma=None, rGrow=0, S=100, r=0),
               CallOption(K=100, T=0.5, price=11.25))

oneyear_IV = hw2analytic.IV(GBMdynamics(sigma=None, rGrow=0, S=100, r=0),
               CallOption(K=100, T=1, price=12))

midpoint_IV = (oneyear_IV + sixmonth_IV) / 2

midpoint_price = hw2analytic.BSpriceCall(GBMdynamics(sigma=midpoint_IV,rGrow=0, S=100, r=0),
                CallOption(K=100, T=.75))

midpoint_price

np.float64(12.081533286249247)

**Answer** <br>
Time-0 price of the .75-expiry call is $12.08

### (c)

The price computed in (b) would allow arbitrage involving the 0.75-expiry call and one of the other contracts. Describe the steps of this arbitrage.

You may assume either cash settlement or physical settlement of these options, but specify what your assumption is.

Cash settlement of an in-the-money option at time $T$ means that you receive $S_T - K$ dollars if you are long the contract, and $K - S_T$ dollars if you are short the contract.

Physical settlement of an in-the-money option at time $T$ means that you receive 1 share of stock and $-K$ dollars if you are long the contract, or $-1$ share of stock and $+K$ dollars if you are short the contract.

Conclusion: expiry interpolation should not be done by linear interpolation of implied volatility. A better alternative would be linear interpolation of the total implied variance $\sigma_{\mathrm{imp}}^2 T$.

**Answer**

I assume cash settlement.

Based on our initial results, I feel confident the .75-expiry call is overpriced as it is greater than the 1-expiry call.

Formally, we know with a fixed strike $K=100$, then call prices must be convex in maturity. Hence, the upperbounnd is the linear interpolation between $C(0.5)$ and $C(1.0)$:

$C(0.75) ≤ 0.5 C(0.5) + 0.5 C(1.0)$

Solving the inequality:

$0.5(11.25) + 0.5(12.00) = 11.625$

From part B) $C(0.75) = 12.08 > 11.625$, so the 0.75 call is overpriced.

We can use this information to construct an arbitrage.

- Short 1 call ($T=0.75$)
- Long 0.5 calls ($T=0.5$)
- Long 0.5 calls ($T=1.0$)

Initial profit $= 12.08 − 11.625 = 0.455 > 0$